# K-Nearest Neighbors: digit classification

This notebook uses the Optical Digits dataset from the `models.md` table to classify handwritten digits (0–9) with KNN.
It explores how feature scaling and the choice of *k* affect performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# ---------- Load and scale ----------
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit scaler on training data only to avoid leakage.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# ---------- KNN (k=5) ----------
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred, average="weighted"):.4f}')
print(f'F1 score:  {f1_score(y_test, y_pred, average="weighted"):.4f}')
print('\nClassification report:\n',
      classification_report(y_test, y_pred,
                            target_names=[str(d) for d in range(10)]))

# ---------- Confusion matrix ----------
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=[str(d) for d in range(10)]
                       ).plot(cmap='Blues')
plt.title('KNN (k=5): confusion matrix')
plt.show()

# ---------- Accuracy vs k ----------
k_range = range(1, 16)
acc_scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    acc_scores.append(accuracy_score(y_test, knn.predict(X_test_sc)))

plt.figure(figsize=(8, 4))
plt.plot(k_range, acc_scores, marker='o', linewidth=2)
plt.xlabel('k (number of neighbors)')
plt.ylabel('Test accuracy')
plt.title('KNN: accuracy vs k')
plt.xticks(list(k_range))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## KNN: model, distance metrics, and evaluation

### Notation

- $\mathbf{x} \in \mathbb{R}^D$: feature vector for a query point ($D=64$ for Optical Digits).
- $\mathcal{N}_k(\mathbf{x})$: the set of $k$ nearest training points to $\mathbf{x}$.
- $y_i \in \{0, 1, \dots, 9\}$: class label of training point $i$.
- $\mathbb{1}[\cdot]$: indicator function (1 if true, 0 otherwise).

### Distance metrics

**Euclidean distance** (default, `metric='minkowski'`, `p=2`):

$$d(\mathbf{x}, \mathbf{x}') = \sqrt{\sum_{j=1}^{D}(x_j - x'_j)^2}$$

**Manhattan distance** (`p=1`):

$$d(\mathbf{x}, \mathbf{x}') = \sum_{j=1}^{D}|x_j - x'_j|$$

**Minkowski distance** (general):

$$d(\mathbf{x}, \mathbf{x}') = \left(\sum_{j=1}^{D}|x_j - x'_j|^p\right)^{1/p}$$

### Decision rule: majority voting

The predicted class is the most common label among the $k$ nearest neighbors:

$$\hat{y} = \operatorname*{argmax}_{c} \sum_{i \in \mathcal{N}_k(\mathbf{x})} \mathbb{1}[y_i = c]$$

### Weighted voting variant

With `weights='distance'`, closer neighbors contribute more:

$$\hat{y} = \operatorname*{argmax}_{c} \sum_{i \in \mathcal{N}_k(\mathbf{x})} \frac{\mathbb{1}[y_i = c]}{d(\mathbf{x}, \mathbf{x}_i)}$$

### Why feature scaling matters

KNN relies on distances. If one feature has a much larger range than others, it dominates the distance calculation. StandardScaler normalizes each feature to zero mean and unit variance so all dimensions contribute equally.

### Key hyperparameters

| Parameter | Typical values | Effect |
| --- | --- | --- |
| `n_neighbors` ($k$) | 1–15 | Small $k$: low bias, high variance; large $k$: high bias, low variance |
| `weights` | `'uniform'`, `'distance'` | Uniform gives equal votes; distance weights by inverse distance |
| `metric` | `'minkowski'`, `'euclidean'`, `'manhattan'` | Distance function used for neighbor lookup |
| `p` | 1 or 2 | Minkowski exponent: 1 = Manhattan, 2 = Euclidean |

### Test-set evaluation

$$\mathrm{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$

$$\mathrm{Precision} = \frac{TP}{TP+FP}, \qquad \mathrm{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2\cdot\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}$$

**Maximize** all four metrics: each ranges from $0$ (worst) to $1$ (best). For multiclass problems, weighted averaging across classes accounts for class imbalance.